<a href="https://colab.research.google.com/github/khalid-alajmi/bayan-applied-nlp-khalid-alajmi/blob/main/notebooks/02_attention_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayan — Applied NLP
## Day 1 — Complete Pipeline

**Student: Khalid Alajmi**

Synthetic bilingual text → Safe preprocessing → Tokenisation → Embeddings → Attention → Transformer Encoder → Evidence


In [1]:
# Bayan — Day 1 Complete Pipeline
# Student: Khalid Alajmi
# Original implementation built for this project from the Day 1 requirements.

# ============================================================
# 0. Runtime
# ============================================================
import importlib.util
import math
import re
import subprocess
import sys
import time
import unicodedata
from dataclasses import dataclass

required = {
    "numpy": "numpy",
    "spacy": "spacy",
    "tokenizers": "tokenizers",
    "transformers": "transformers",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import spacy
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing

RANDOM_SEED = 17
random = np.random.default_rng(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("DAY1_RUNTIME=READY")

# ============================================================
# 1. Synthetic bilingual input + Unicode
# ============================================================
feedback = [
    "طلب تحديث التصريح ما زال قيد المعالجة.",
    "الخدمــة ممتازة\u00a0وسريعة، شكراً لكم.",
    "Please contact demo.user@example.invalid for support.",
    "رقم التواصل التجريبي 0558764321",
    "The service request was completed successfully.",
]

def unicode_profile(text):
    return [
        (character, f"U+{ord(character):04X}", unicodedata.name(character, "UNKNOWN"))
        for character in text
    ]

unicode_example = unicode_profile("أé")
assert unicode_example[0][1] == "U+0623"
print("Unicode inspection=PASS")

# ============================================================
# 2. Privacy-first, two-copy preprocessing
# ============================================================
EMAIL_PATTERN = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_PATTERN = re.compile(r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)")
SPACE_PATTERN = re.compile(r"\s+")
TATWEEL = "\u0640"

@dataclass(frozen=True)
class PreparedText:
    raw_text: str
    display_text: str
    model_text: str

def protect_private_text(text):
    protected = EMAIL_PATTERN.sub("<EMAIL>", text)
    return PHONE_PATTERN.sub("<PHONE>", protected)

def prepare_for_bayan(text):
    if not isinstance(text, str):
        raise TypeError("Bayan input must be text.")

    raw = text
    normalized = unicodedata.normalize("NFC", text)
    normalized = normalized.replace(TATWEEL, "")
    normalized = SPACE_PATTERN.sub(" ", normalized).strip()

    safe = protect_private_text(normalized)
    return PreparedText(
        raw_text=raw,
        display_text=safe,
        model_text=safe,
    )

prepared_feedback = [prepare_for_bayan(item) for item in feedback]

assert prepared_feedback[2].raw_text == feedback[2]
assert "demo.user@example.invalid" not in prepared_feedback[2].model_text
assert "0558764321" not in prepared_feedback[3].model_text
assert "<EMAIL>" in prepared_feedback[2].model_text
assert "<PHONE>" in prepared_feedback[3].model_text
assert TATWEEL not in prepared_feedback[1].model_text

print("Two-copy privacy contract=PASS")
print("Protected examples:")
for item in prepared_feedback:
    print(" -", item.model_text)

# ============================================================
# 3. Sentence segmentation
# ============================================================
segmenter = spacy.blank("xx")
segmenter.add_pipe("sentencizer")

def sentence_split(text):
    safe_text = prepare_for_bayan(text).model_text
    document = segmenter(safe_text)
    return [sentence.text.strip() for sentence in document.sents if sentence.text.strip()]

segmentation_tests = {
    "ar": sentence_split("تم استلام الطلب. ما زلت أنتظر التحديث!"),
    "en": sentence_split("The request was received. I am waiting for an update."),
}
abbreviation_probe = sentence_split("راجع د. أحمد. ثم أعد المحاولة.")

assert len(segmentation_tests["ar"]) == 2
assert len(segmentation_tests["en"]) == 2

print("Arabic sentences:", segmentation_tests["ar"])
print("English sentences:", segmentation_tests["en"])
print("Known abbreviation limitation:", abbreviation_probe)
print("Sentence segmentation=PASS")

# ============================================================
# 4. Local WordPiece demonstration
# ============================================================
SPECIAL = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
PROJECT_VOCAB = SPECIAL + [
    "طلب", "تحديث", "التصريح", "ما", "زال", "قيد", "المعالجة", ".",
    "الخدمة", "ممتازة", "وسريعة", "شكراً", "لكم", "،",
    "Please", "contact", "for", "support", "The", "service", "request",
    "was", "completed", "successfully", "<", "EMAIL", ">", "رقم",
    "التواصل", "التجريبي", "PHONE",
]
token_to_id = {token: index for index, token in enumerate(PROJECT_VOCAB)}

local_wp = Tokenizer(WordPiece(vocab=token_to_id, unk_token="[UNK]"))
local_wp.pre_tokenizer = BertPreTokenizer()
local_wp.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[
        ("[CLS]", token_to_id["[CLS]"]),
        ("[SEP]", token_to_id["[SEP]"]),
    ],
)

demo_encoding = local_wp.encode("طلب تحديث التصريح")
assert demo_encoding.tokens[0] == "[CLS]"
assert demo_encoding.tokens[-1] == "[SEP]"

print("Local tokens:", demo_encoding.tokens)
print("Local token IDs:", demo_encoding.ids)
print("Local WordPiece=PASS")

# ============================================================
# 5. Token fertility + truncation diagnostics
# ============================================================
def approximate_words(text):
    return max(1, len(text.split()))

def local_content_tokens(text):
    encoded = local_wp.encode(text)
    return [token for token in encoded.tokens if token not in {"[CLS]", "[SEP]", "[PAD]"}]

def fertility(text):
    return len(local_content_tokens(text)) / approximate_words(text)

def would_truncate(text, limit):
    return len(local_wp.encode(text).ids) > limit

model_texts = [item.model_text for item in prepared_feedback]
local_fertilities = [fertility(text) for text in model_texts]
local_mean_fertility = float(np.mean(local_fertilities))
local_truncation_10 = float(np.mean([would_truncate(text, 10) for text in model_texts]))

assert all(value > 0 for value in local_fertilities)
assert 0 <= local_truncation_10 <= 1

print("Local fertility per sample:", [round(x, 2) for x in local_fertilities])
print("Local mean fertility:", round(local_mean_fertility, 2))
print("Local truncation @10:", f"{local_truncation_10:.0%}")
print("Token diagnostics=PASS")

# ============================================================
# 6. Padding, attention mask, and embeddings
# ============================================================
BATCH_LENGTH = 12
local_wp.enable_truncation(max_length=BATCH_LENGTH)
local_wp.enable_padding(
    length=BATCH_LENGTH,
    pad_id=token_to_id["[PAD]"],
    pad_token="[PAD]",
)

encoded_batch = [local_wp.encode(text) for text in model_texts[:2]]
input_ids = np.asarray([row.ids for row in encoded_batch], dtype=np.int64)
padding_mask = np.asarray([row.attention_mask for row in encoded_batch], dtype=np.int64)

EMBEDDING_SIZE = 8
embedding_matrix = random.normal(
    loc=0.0,
    scale=0.02,
    size=(len(PROJECT_VOCAB), EMBEDDING_SIZE),
)
token_embeddings = embedding_matrix[input_ids]

assert input_ids.shape == (2, BATCH_LENGTH)
assert padding_mask.shape == input_ids.shape
assert token_embeddings.shape == (2, BATCH_LENGTH, EMBEDDING_SIZE)

print("input_ids:", input_ids.shape)
print("attention_mask:", padding_mask.shape)
print("embeddings:", token_embeddings.shape)
print("IDs → embeddings=PASS")

# Return tokenizer to unpadded form before comparisons.
local_wp.no_padding()
local_wp.no_truncation()

# ============================================================
# 7. Explore / Distinction tokenizer comparison
# ============================================================
comparison_texts = [
    "طلب تجديد التصريح ما زال قيد المعالجة.",
    "الخدمة ممتازة وتم إنجاز الطلب بسرعة.",
    "لم أستلم تحديثاً عن طلبي حتى الآن.",
    "The service request was completed successfully.",
    "I am still waiting for an update on my request.",
]

tokenizer_evidence = {}

try:
    from transformers import AutoTokenizer

    multilingual_tokenizers = {
        "mBERT": "google-bert/bert-base-multilingual-cased",
        "XLM-R": "FacebookAI/xlm-roberta-base",
    }

    for label, model_id in multilingual_tokenizers.items():
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
        protected = [prepare_for_bayan(text).model_text for text in comparison_texts]

        fertility_values = [
            len(tokenizer.tokenize(text)) / approximate_words(text)
            for text in protected
        ]
        truncation_values = [
            len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"]) > 10
            for text in protected
        ]

        start = time.perf_counter()
        repetitions = 50
        for _ in range(repetitions):
            for text in protected:
                tokenizer.tokenize(text)
        elapsed_ms = (
            (time.perf_counter() - start)
            / (repetitions * len(protected))
            * 1000
        )

        tokenizer_evidence[label] = {
            "fertility": float(np.mean(fertility_values)),
            "truncation": float(np.mean(truncation_values)),
            "time_ms": elapsed_ms,
        }

        print(
            f"{label}: fertility={tokenizer_evidence[label]['fertility']:.2f}, "
            f"truncation@10={tokenizer_evidence[label]['truncation']:.0%}, "
            f"time={elapsed_ms:.3f} ms"
        )

    print("EXPLORE_TOKENIZER_COMPARISON=PASS")
    print("DISTINCTION_TOKENIZER_COMPARISON=PASS")

except Exception as error:
    print("Tokenizer comparison needs network access in this run.")
    print("Reason:", type(error).__name__)
    print("Core pipeline is unaffected.")

# ============================================================
# 8. Scaled dot-product attention written for Bayan
# ============================================================
def stable_softmax(values, axis=-1):
    centered = values - np.max(values, axis=axis, keepdims=True)
    exponentials = np.exp(centered)
    return exponentials / np.sum(exponentials, axis=axis, keepdims=True)

def bayan_attention(query, key, value, allowed_keys=None):
    query = np.asarray(query, dtype=float)
    key = np.asarray(key, dtype=float)
    value = np.asarray(value, dtype=float)

    if query.shape[-1] != key.shape[-1]:
        raise ValueError("Query and key dimensions must match.")
    if key.shape[-2] != value.shape[-2]:
        raise ValueError("Key and value sequence lengths must match.")

    logits = query @ np.swapaxes(key, -1, -2)
    logits = logits / math.sqrt(query.shape[-1])

    if allowed_keys is not None:
        allowed_keys = np.asarray(allowed_keys, dtype=bool)
        allowed_keys = np.broadcast_to(allowed_keys, logits.shape)
        if np.any(allowed_keys.sum(axis=-1) == 0):
            raise ValueError("Every query must be allowed to attend to at least one key.")
        logits = np.where(allowed_keys, logits, -1e9)

    weights = stable_softmax(logits, axis=-1)
    context = weights @ value
    return context, weights

Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
V = np.array([[8.0, 0.0], [0.0, 8.0], [4.0, 4.0]])

attention_output, attention_weights = bayan_attention(Q, K, V)

assert attention_output.shape == (2, 2)
assert np.allclose(attention_weights.sum(axis=-1), 1.0)

print("Attention weights:")
print(np.round(attention_weights, 3))
print("Attention output:")
print(np.round(attention_output, 3))
print("Scaled dot-product attention=PASS")

# ============================================================
# 9. Scaling check
# ============================================================
def mean_attention_entropy(probabilities):
    safe = np.clip(probabilities, 1e-12, 1.0)
    return float(np.mean(-np.sum(safe * np.log(safe), axis=-1)))

large_q = random.normal(size=(6, 64))
large_k = random.normal(size=(6, 64))
unscaled = stable_softmax(large_q @ large_k.T)
scaled = stable_softmax((large_q @ large_k.T) / math.sqrt(64))

print("Entropy without scaling:", round(mean_attention_entropy(unscaled), 3))
print("Entropy with scaling:", round(mean_attention_entropy(scaled), 3))
print("Scaling check=PASS")

# ============================================================
# 10. Mask semantics: True=keep, False=block
# ============================================================
keep = np.array([
    [True,  False, False],
    [True,  True,  False],
])

masked_context, masked_weights = bayan_attention(Q, K, V, allowed_keys=keep)

assert np.allclose(masked_weights[~keep], 0.0, atol=1e-12)
assert np.allclose(masked_weights.sum(axis=-1), 1.0)

print("Masked weights:")
print(np.round(masked_weights, 3))
print("Attention masking=PASS")

# ============================================================
# 11. Multi-head shape journey
# ============================================================
def to_heads(tensor, number_of_heads):
    batch_size, sequence_length, model_width = tensor.shape
    if model_width % number_of_heads != 0:
        raise ValueError("Model width must be divisible by the number of heads.")
    head_width = model_width // number_of_heads
    reshaped = tensor.reshape(
        batch_size, sequence_length, number_of_heads, head_width
    )
    return reshaped.transpose(0, 2, 1, 3)

def from_heads(head_tensor):
    batch_size, number_of_heads, sequence_length, head_width = head_tensor.shape
    reordered = head_tensor.transpose(0, 2, 1, 3)
    return reordered.reshape(
        batch_size, sequence_length, number_of_heads * head_width
    )

encoder_input = random.normal(size=(2, 5, 12))
head_view = to_heads(encoder_input, number_of_heads=3)
restored_input = from_heads(head_view)

assert head_view.shape == (2, 3, 5, 4)
assert np.allclose(restored_input, encoder_input)

print("Encoder input:", encoder_input.shape)
print("Split heads:", head_view.shape)
print("Recombined:", restored_input.shape)
print("Multi-head shapes=PASS")

# ============================================================
# 12. Transformer encoder block: real forward pass
# ============================================================
try:
    import torch

    torch.manual_seed(RANDOM_SEED)

    encoder_layer = torch.nn.TransformerEncoderLayer(
        d_model=12,
        nhead=3,
        dim_feedforward=24,
        dropout=0.0,
        batch_first=True,
    )
    encoder_layer.eval()

    encoder_tensor = torch.tensor(encoder_input, dtype=torch.float32)
    with torch.no_grad():
        encoder_result = encoder_layer(encoder_tensor)

    assert tuple(encoder_result.shape) == (2, 5, 12)
    print("Transformer encoder output:", tuple(encoder_result.shape))
    print("TRANSFORMER_ENCODER_FORWARD=PASS")

    # Independent parity check for our small attention example.
    import torch.nn.functional as F

    tq = torch.tensor(Q, dtype=torch.float64)[None, None]
    tk = torch.tensor(K, dtype=torch.float64)[None, None]
    tv = torch.tensor(V, dtype=torch.float64)[None, None]

    torch_attention = F.scaled_dot_product_attention(
        tq, tk, tv, dropout_p=0.0
    ).numpy()[0, 0]

    parity_error = float(np.max(np.abs(torch_attention - attention_output)))
    assert parity_error < 1e-9
    print("NumPy/PyTorch attention difference:", f"{parity_error:.3e}")
    print("ATTENTION_PARITY=PASS")

except ModuleNotFoundError:
    raise RuntimeError("PyTorch is required for the Day 1 encoder evidence.")

# ============================================================
# 13. Optional real multilingual checkpoint evidence
# ============================================================
real_model_evidence = False

try:
    from transformers import AutoModel, AutoTokenizer

    REAL_MODEL = "distilbert/distilbert-base-multilingual-cased"
    real_tokenizer = AutoTokenizer.from_pretrained(REAL_MODEL, use_fast=True)
    real_model = AutoModel.from_pretrained(
        REAL_MODEL,
        attn_implementation="eager",
    )
    real_model.eval()

    real_examples = [
        "الخدمة لم تتأخر.",
        "The service was not delayed.",
    ]
    real_batch = real_tokenizer(
        real_examples,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt",
    )

    with torch.no_grad():
        real_forward = real_model(**real_batch, output_attentions=True)

    real_hidden_shape = tuple(real_forward.last_hidden_state.shape)
    real_attention_shape = tuple(real_forward.attentions[0].shape)
    parameter_count = sum(p.numel() for p in real_model.parameters())

    assert real_hidden_shape[:2] == tuple(real_batch["input_ids"].shape)
    assert real_attention_shape[0] == len(real_examples)
    assert real_attention_shape[-1] == real_attention_shape[-2]
    assert parameter_count > 100_000_000

    print("Real checkpoint:", REAL_MODEL)
    print("Parameters:", f"{parameter_count:,}")
    print("Hidden states:", real_hidden_shape)
    print("Attention [batch, heads, query, key]:", real_attention_shape)
    print("ACTUAL_TRANSFORMER_FORWARD=PASS")
    real_model_evidence = True

except Exception as error:
    print("Real checkpoint evidence requires model download/network.")
    print("Reason:", type(error).__name__)

# ============================================================
# 14. Day 1 verification
# ============================================================
notebook1_checks = {
    "unicode": unicode_example[0][1] == "U+0623",
    "privacy": (
        "<EMAIL>" in prepared_feedback[2].model_text
        and "<PHONE>" in prepared_feedback[3].model_text
    ),
    "raw_preserved": prepared_feedback[2].raw_text == feedback[2],
    "sentence_stage": all(len(value) == 2 for value in segmentation_tests.values()),
    "token_metrics": (
        all(value > 0 for value in local_fertilities)
        and 0 <= local_truncation_10 <= 1
    ),
    "embedding_shape": token_embeddings.shape == (2, 12, 8),
}

notebook2_checks = {
    "attention_rows_sum_to_one": np.allclose(attention_weights.sum(axis=-1), 1.0),
    "masked_positions_are_zero": np.allclose(masked_weights[~keep], 0.0, atol=1e-12),
    "attention_output_shape": attention_output.shape == (2, 2),
    "multihead_round_trip": np.allclose(restored_input, encoder_input),
    "encoder_shape": tuple(encoder_result.shape) == (2, 5, 12),
    "attention_parity": parity_error < 1e-9,
}

print("\n=== NOTEBOOK 01 CORE ===")
for name, passed in notebook1_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(notebook1_checks.values())
print("DAY1_NOTEBOOK1_CORE=PASS")

print("\n=== NOTEBOOK 02 CORE ===")
for name, passed in notebook2_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(notebook2_checks.values())
print("DAY1_NOTEBOOK2_CORE=PASS")

# ============================================================
# 15. Gate A evidence
# ============================================================
print("\n=== DAY 1 — GATE A EVIDENCE ===")
print("Preprocessing profile:")
print("  NFC + whitespace normalization + tatweel removal + PII masking")
print("Privacy example:")
print("  0558764321 -> <PHONE>")
print("Local mean token fertility:")
print(" ", round(local_mean_fertility, 2))
print("Local truncation rate @10:")
print(" ", f"{local_truncation_10:.0%}")
print("Tokenizer decision:")
print(
    "  Use conservative preprocessing and measure tokenisation on bilingual "
    "examples before selecting the final model tokenizer. Fertility, "
    "truncation, and timing are diagnostics, not a universal quality ranking."
)
print("Attention decision:")
print(
    "  Encoder padding positions must be blocked from attention. In this "
    "implementation True means keep and False means block; valid attention "
    "rows are verified to sum to one."
)
print("Known limitation:")
print(
    "  The rule-based sentence segmenter needs additional handling for "
    "abbreviations before production use."
)

if tokenizer_evidence:
    for label, evidence in tokenizer_evidence.items():
        print(
            f"{label}: fertility={evidence['fertility']:.2f}, "
            f"truncation@10={evidence['truncation']:.0%}, "
            f"time={evidence['time_ms']:.3f} ms"
        )

if real_model_evidence:
    print("Real multilingual transformer evidence: PASS")
else:
    print("Real multilingual transformer evidence: NOT MEASURED IN THIS RUN")

print("DAY1_GATE_A_TECHNICAL_EVIDENCE=PASS")
print("\nDAY1_PIPELINE_EXECUTION=COMPLETE")


Python: 3.13.15
DAY1_RUNTIME=READY
Unicode inspection=PASS
Two-copy privacy contract=PASS
Protected examples:
 - طلب تحديث التصريح ما زال قيد المعالجة.
 - الخدمة ممتازة وسريعة، شكراً لكم.
 - Please contact <EMAIL> for support.
 - رقم التواصل التجريبي <PHONE>
 - The service request was completed successfully.
Arabic sentences: ['تم استلام الطلب.', 'ما زلت أنتظر التحديث!']
English sentences: ['The request was received.', 'I am waiting for an update.']
Known abbreviation limitation: ['راجع د.', 'أحمد.', 'ثم أعد المحاولة.']
Sentence segmentation=PASS
Local tokens: ['[CLS]', 'طلب', 'تحديث', 'التصريح', '[SEP]']
Local token IDs: [2, 5, 6, 7, 3]
Local WordPiece=PASS
Local fertility per sample: [1.14, 1.4, 1.6, 1.5, 1.17]
Local mean fertility: 1.36
Local truncation @10: 0%
Token diagnostics=PASS
input_ids: (2, 12)
attention_mask: (2, 12)
embeddings: (2, 12, 8)
IDs → embeddings=PASS


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Real checkpoint: distilbert/distilbert-base-multilingual-cased
Parameters: 134,734,080
Hidden states: (2, 10, 768)
Attention [batch, heads, query, key]: (2, 12, 10, 10)
ACTUAL_TRANSFORMER_FORWARD=PASS

=== NOTEBOOK 01 CORE ===
unicode: PASS
privacy: PASS
raw_preserved: PASS
sentence_stage: PASS
token_metrics: PASS
embedding_shape: PASS
DAY1_NOTEBOOK1_CORE=PASS

=== NOTEBOOK 02 CORE ===
attention_rows_sum_to_one: PASS
masked_positions_are_zero: PASS
attention_output_shape: PASS
multihead_round_trip: PASS
encoder_shape: PASS
attention_parity: PASS
DAY1_NOTEBOOK2_CORE=PASS

=== DAY 1 — GATE A EVIDENCE ===
Preprocessing profile:
  NFC + whitespace normalization + tatweel removal + PII masking
Privacy example:
  0558764321 -> <PHONE>
Local mean token fertility:
  1.36
Local truncation rate @10:
  0%
Tokenizer decision:
  Use conservative preprocessing and measure tokenisation on bilingual examples before selecting the final model tokenizer. Fertility, truncation, and timing are diagnostics,